# RoadRisk AI - Optional Dashcam Video Inference Demo

**Purpose:** Demonstrate the perception layer without training a model from scratch.

This notebook uses a pretrained Ultralytics YOLO detector/tracker on a short uploaded road video. It is an **MVP feasibility demo**, not the final near-miss classifier.

### Why this is useful
- Shows that vehicles, motorcycles, buses/trucks and pedestrians can be detected/tracked.
- Produces a basic object-count/time graph.
- Lets you inspect Sri Lankan domain-shift failures before deciding whether fine-tuning is necessary.

### Colab setting
Use **Runtime -> Change runtime type -> T4 GPU** if available.


In [ ]:
# Install the detector/tracker package.
!pip -q install ultralytics

import os
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from ultralytics import YOLO

OUTPUT_DIR='/content/roadrisk_video_demo'
os.makedirs(OUTPUT_DIR,exist_ok=True)


## 1. Upload a short dashcam video

Use a legally obtained clip that you are allowed to analyse. For your own road footage, mount the phone/camera securely and do not interact with it while driving.

In [ ]:
uploaded=files.upload()
video_file=next(iter(uploaded))
print('Uploaded:',video_file)


## 2. Run pretrained tracking

This does **not** train anything. The model is used as a baseline. If it performs poorly on three-wheelers, motorcycles, night scenes or dense local traffic, that becomes evidence for targeted fine-tuning later.

In [ ]:
model=YOLO('yolo11n.pt')

# COCO road-user class IDs commonly useful to RoadRisk:
# 0 person, 1 bicycle, 2 car, 3 motorcycle, 5 bus, 7 truck
road_classes=[0,1,2,3,5,7]

results=model.track(
    source=video_file,
    classes=road_classes,
    persist=True,
    save=True,
    project=OUTPUT_DIR,
    name='tracked',
    conf=0.25,
    verbose=False
)

print('Tracking finished. Output folder:',os.path.join(OUTPUT_DIR,'tracked'))


## 3. Generate a simple detections-over-time graph

This is not a near-miss metric. It only helps inspect scene density and whether the detector is functioning.

In [ ]:
cap=cv2.VideoCapture(video_file)
fps=cap.get(cv2.CAP_PROP_FPS) or 30.0
cap.release()

rows=[]
for i,res in enumerate(results):
    boxes=res.boxes
    counts={}
    if boxes is not None and boxes.cls is not None:
        for cls_id in boxes.cls.cpu().numpy().astype(int):
            name=model.names.get(cls_id,str(cls_id))
            counts[name]=counts.get(name,0)+1
    rows.append({'time_s':i/fps,'total_objects':sum(counts.values()),**counts})

count_df=pd.DataFrame(rows).fillna(0)
display(count_df.head())
count_df.to_csv(os.path.join(OUTPUT_DIR,'frame_object_counts.csv'),index=False)

ax=count_df.plot(x='time_s',y='total_objects',legend=False)
ax.set_title('Detected Road Users Over Time - Baseline Demo')
ax.set_xlabel('Video time (s)')
ax.set_ylabel('Detected road users per frame')
plt.tight_layout()
fig_path=os.path.join(OUTPUT_DIR,'fig_10_baseline_object_density.png')
plt.savefig(fig_path,dpi=220,bbox_inches='tight')
plt.show()
print('Saved:',fig_path)


## 4. What to inspect manually

For 20-50 representative frames, record whether the pretrained model:

- misses motorcycles or pedestrians,
- confuses three-wheelers with cars,
- loses tracks during occlusion,
- fails at dusk/night/rain,
- produces unstable IDs in dense traffic.

Only after you measure these failures should you decide whether to fine-tune a detector on IDD/I2WDD or a local dataset.

**Do not claim this notebook detects near-misses.** The final near-miss layer requires trajectories plus temporal/event features and validation.